# M0.1 · Vektör, iç çarpım, norm

Lineer cebirin atomu olan iç çarpımı ve ondan türeyen norm ile kosinüs benzerliğini 3 elemanlı vektörlerde elle, sonra döngüyle, NumPy'da bir benzerlik matrisi olarak, PyTorch'ta gradyanıyla ve en sonda binlerce çift için tek çağrıda hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m0-1-dot.html](https://m-gonul.github.io/pytorch-lab/m0-1-dot.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m0_1_dot.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need\n\n\ndef raises(f, exc=ValueError):\n    try:\n        f()\n    except exc:\n        return\n    raise AssertionError(f\"{exc.__name__} bekleniyordu\")\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"pandas\"}))\n    for n in (\"dot\", \"norm\", \"cosine\"):\n        R.test(f\"`{n}` tanımlı\", lambda n=n: need(ns, n))\n    if not all(n in ns for n in (\"dot\", \"norm\", \"cosine\")):\n        return\n    dot, norm, cosine = ns[\"dot\"], ns[\"norm\"], ns[\"cosine\"]\n    R.test(\"elle örnek: u·v = 12\", lambda: close(dot([1, 2, 3], [4, -5, 6]), 12))\n    R.test(\"‖u‖ = √14\", lambda: close(norm([1, 2, 3]), math.sqrt(14)))\n    R.test(\"cos θ = 12/√1078\", lambda: close(cosine([1, 2, 3], [4, -5, 6]), 12 / math.sqrt(1078)))\n\n    def rnd():\n        rng = random.Random(1)\n        for d in (1, 5, 17):\n            u = [rng.uniform(-3, 3) for _ in range(d)]\n            v = [rng.uniform(-3, 3) for _ in range(d)]\n            close(dot(u, v), sum(a * b for a, b in zip(u, v)), what=f\"d={d} dot\")\n            c = cosine(u, v)\n            assert -1 - 1e-12 <= c <= 1 + 1e-12, f\"kosinüs [-1,1] dışında: {c}\"\n    R.test(\"rastgele vektörler (d = 1, 5, 17)\", rnd)\n    R.test(\"kendisiyle kosinüs = 1, zıttıyla = −1\", lambda: (close(cosine([2, 1], [4, 2]), 1), close(cosine([2, 1], [-2, -1]), -1)))\n    R.test(\"farklı uzunlukta ValueError\", lambda: raises(lambda: dot([1, 2], [1, 2, 3])))\n    R.test(\"sıfır vektörde ValueError\", lambda: raises(lambda: cosine([0, 0], [1, 2])))\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü/comprehension yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(3)\n    X = rng.normal(size=(9, 5))\n    Xn = X / np.sqrt((X ** 2).sum(1))[:, None]\n    ref = Xn @ Xn.T\n\n    def cm():\n        S = need(ns, \"cosine_matrix\")(X)\n        close(S, ref, what=\"benzerlik matrisi\")\n        close(np.diag(S), np.ones(9), what=\"köşegen (kendisiyle benzerlik)\")\n        close(S, np.asarray(S).T, what=\"simetri\")\n    R.test(\"cosine_matrix doğru, köşegen 1, simetrik\", cm)\n    R.test(\"elle örnek u, v için S[0,1] = 0.3655\", lambda: close(need(ns, \"cosine_matrix\")(np.array([[1., 2, 3], [4, -5, 6]]))[0, 1], 12 / np.sqrt(1078)))\n\n    def ms():\n        f = need(ns, \"most_similar\")\n        for i in range(9):\n            r = ref[i].copy(); r[i] = -np.inf\n            got = f(X, i)\n            assert int(got) == int(np.argmax(r)), f\"i={i}: {got} bulundu, {int(np.argmax(r))} bekleniyordu\"\n        T = np.array([[1.0, 0.0], [0.9, 0.1], [0.0, 1.0], [-1.0, 0.0]])\n        assert int(f(T, 0)) == 1, \"satır 0 kendisini seçmemeli\"\n    R.test(\"most_similar kendisini dışarıda bırakıyor\", ms)\n",
 "d": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, uses_any, ensure\n\n\ndef check(ns, R):\n    u = torch.tensor([1., 2., 3.], dtype=torch.float64)\n    v = torch.tensor([4., -5., 6.], dtype=torch.float64)\n    g = torch.Generator().manual_seed(0)\n    a, b = torch.randn(7, generator=g, dtype=torch.float64), torch.randn(7, generator=g, dtype=torch.float64)\n\n    def cos():\n        f = need(ns, \"cosine_torch\")\n        close(f(u, v), 12 / 1078 ** 0.5)\n        close(f(a, b), F.cosine_similarity(a, b, dim=0), what=\"rastgele 7-boyutlu\")\n    R.test(\"cosine_torch doğru\", cos)\n\n    def grad():\n        gr = need(ns, \"cosine_grad\")\n        for x, y in ((u, v), (a, b)):\n            got = gr(x, y)\n            c = (x @ y) / (x.norm() * y.norm())\n            want = y / (x.norm() * y.norm()) - c * x / x.norm() ** 2\n            close(got, want, what=\"∂cos/∂u\")\n            ensure(abs(float(got @ x)) < 1e-9, f\"gradyan u'ya dik değil: g·u = {float(got @ x):.3g}\")\n        return \"formülle eşleşti ve gradyan u'ya dik (g·u ≈ 0)\"\n    R.test(\"cosine_grad formülle eşleşiyor, u'ya dik\", grad)\n    R.test(\"autograd kullanılıyor\", lambda: ensure(uses_any(ns[\"__source__\"], [\"backward\", \"grad\"]), \"gradyanı elle yazma; .backward() kullan\"))\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, no_loops, need, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    U, V = torch.randn(64, 16, generator=g), torch.randn(64, 16, generator=g)\n    Q, K = torch.randn(10, 16, generator=g), torch.randn(30, 16, generator=g)\n    R.test(\"rowwise_dot doğru\", lambda: close(need(ns, \"rowwise_dot\")(U, V), (U * V).sum(1), rtol=1e-5, atol=1e-5))\n    R.test(\"pairwise_cosine doğru, shape (n, m)\", lambda: close(need(ns, \"pairwise_cosine\")(Q, K), F.cosine_similarity(Q[:, None], K[None], dim=-1), rtol=1e-4, atol=1e-5))\n    if \"rowwise_dot\" not in ns:\n        return\n    rd = ns[\"rowwise_dot\"]\n    A, B = torch.randn(4096, 64), torch.randn(4096, 64)\n    t_loop = bench(lambda: torch.stack([torch.dot(a, b) for a, b in zip(A, B)]), repeat=3)\n    t_vec = bench(lambda: rd(A, B))\n    t_mul = bench(lambda: (A * B).sum(1))\n    R.table(\"4096 çiftin iç çarpımı (d = 64)\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for + torch.dot\", fmt_time(t_loop), \"1×\"],\n             [\"senin rowwise_dot\", fmt_time(t_vec), f\"{t_loop / t_vec:,.0f}× hızlı\"],\n             [\"(A * B).sum(1)\", fmt_time(t_mul), f\"{t_loop / t_mul:,.0f}× hızlı\"]],\n            note=\"(A * B).sum(1) önce (4096, 64) ara tensörü yazar; einsum bunu atlayabilir. Farkı belleğe yazılan bayt sayısı belirler.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m0_1_dot as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

İki vektörün shape'i aynıysa iç çarpımları tek bir sayıdır:

$$
u\cdot v=\sum_{i=1}^{d}u_i v_i = \lVert u\rVert\,\lVert v\rVert\cos\theta .
$$

Bu iki tanım arasındaki köprü ML'in her yerinde: bir nöronun ön-aktivasyonu $w\cdot x$, bir attention skoru $q\cdot k$, bir embedding araması kosinüs benzerliği. Norm, vektörün kendisiyle iç çarpımının kökü: $\lVert u\rVert=\sqrt{u\cdot u}$.

Bu modülde aynı hesabı beş kez yazacaksın. Her seferinde bir öncekinin sonucuyla eşleşmesi gerekiyor; değişen tek şey hesabı **kimin** ve **kaç eleman için birden** yaptığı.

## 1 · Elle

$$
u=\begin{bmatrix}1\\2\\3\end{bmatrix},\qquad v=\begin{bmatrix}4\\-5\\6\end{bmatrix}
$$

Önce üç eleman çarpımını tek tek yaz, sonra topla. Normu hesaplarken kökü tahmin et: hangi iki tam sayının karesi 14'ü sıkıştırıyor? Ondalık kısmı lineer aradeğerle bul ($\sqrt{14}\approx 3.7+\frac{14-13.69}{2\cdot3.7}$); bu, Newton yönteminin tek adımıdır.

Ondalık ayırıcı olarak nokta ya da virgül kullanabilirsin; kesir de olur (`12/32.83`).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$u=[1,\,2,\,3]$, $v=[4,\,-5,\,6]$: eleman çarpımları ve iç çarpım**

- `p1`: $u_1v_1$
- `p2`: $u_2v_2$
- `p3`: $u_3v_3$
- `dot`: $u\cdot v$

**Normlar ve açı**

- `nu2`: $\lVert u\rVert^2 = u\cdot u$
- `nv2`: $\lVert v\rVert^2$
- `nu`: $\lVert u\rVert$ (2 ondalık)
- `cos`: $\cos\theta$ (3 ondalık)
- `acute`: Açı dar mı? (dar: 1, geniş: 0)

In [ ]:
cevaplar = dict(
    p1=None,
    p2=None,
    p3=None,
    dot=None,
    nu2=None,
    nv2=None,
    nu=None,
    cos=None,
    acute=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"p1":[4,0],"p2":[-10,0],"p3":[18,0],"dot":[12,0],"nu2":[14,0],"nv2":[77,0],"nu":[3.7417,0.006],"cos":[0.36549,0.0012],"acute":[1,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Listelerle üç fonksiyon yaz; `numpy`, `torch` ve `math.dist`/`math.hypot` gibi kısayollar yok, ama `math.sqrt` serbest.

- `dot(u, v)` — uzunluklar farklıysa `ValueError`.
- `norm(u)` — `dot`'u kullanarak.
- `cosine(u, v)` — sıfır vektör için tanımsız; bu durumda `ValueError` fırlat (`if` burada işe yarıyor).

In [ ]:
%%writefile stage_b.py
import math


def dot(u, v):
    ...


def norm(u):
    ...


def cosine(u, v):
    ...


print(dot([1, 2, 3], [4, -5, 6]))      # 12
print(norm([1, 2, 3]))                 # 3.7416...
print(cosine([1, 2, 3], [4, -5, 6]))   # 0.3654...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da tek çift için `u @ v` ve `np.linalg.norm(u)` yeter; asıl kazanç **çok çift**te. Satırları vektör olan bir `X` (`n × d`) için tüm çiftlerin kosinüs benzerliği:

$$
\hat X = \frac{X}{\lVert X\rVert_{\text{satır}}},\qquad S=\hat X\hat X^\top\quad (n\times n).
$$

Bölme broadcasting ile olur: normlar `(n, 1)` shape'inde tutulursa `(n, d)` ile sağdan hizalanır. `keepdims=True` tam bunun için var.

- `cosine_matrix(X)` — `(n, n)` benzerlik matrisi. **Döngü yok.**
- `most_similar(X, i)` — `i`. satıra en benzer *başka* satırın indeksi (kendisi hariç). İpucu: köşegeni `-inf` yap, `argmax` al.

In [ ]:
%%writefile stage_c.py
import numpy as np


def cosine_matrix(X):
    """X: (n, d) → (n, n) kosinüs benzerlik matrisi. Döngü yok."""
    ...


def most_similar(X, i):
    """i. satıra en benzer başka satırın indeksi."""
    ...


X = np.array([[1.0, 0.0], [0.9, 0.1], [0.0, 1.0], [-1.0, 0.0]])
print(np.round(cosine_matrix(X), 3))
print(most_similar(X, 0))  # 1

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Kosinüsün $u$'ya göre gradyanı, embedding'leri eğitirken (ör. contrastive loss) ağın gördüğü sinyaldir. Bölüm kuralıyla:

$$
\frac{\partial \cos\theta}{\partial u}=\frac{v}{\lVert u\rVert\lVert v\rVert}-\cos\theta\,\frac{u}{\lVert u\rVert^2}.
$$

İkinci terim gradyanın $u$ yönündeki bileşenini siler: uzunluğu değiştirmek kosinüsü değiştirmez, yani gradyan $u$'ya **diktir**. Kontrol bunu da doğrular.

- `cosine_torch(u, v)` — tensörlerle kosinüs.
- `cosine_grad(u, v)` — autograd ile $\partial\cos\theta/\partial u$.

In [ ]:
%%writefile stage_d.py
import torch


def cosine_torch(u, v):
    ...


def cosine_grad(u, v):
    """∂cos(u, v)/∂u — autograd ile."""
    u = u.detach().clone().requires_grad_(True)
    ...


u = torch.tensor([1., 2., 3.])
v = torch.tensor([4., -5., 6.])
print(cosine_torch(u, v))
print(cosine_grad(u, v))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Gerçek kullanımda bir sorgu kümesi `Q` (`n × d`) ile bir veri tabanı `K` (`m × d`) arasındaki tüm benzerlikler, ya da `n` vektör çiftinin satır satır iç çarpımları gerekir. İkisi de tek `einsum`:

```python
torch.einsum("nd,nd->n", U, V)   # satır satır iç çarpım: (U * V).sum(-1)
torch.einsum("nd,md->nm", Q, K)  # tüm çiftler: Q @ K.T
```

- `rowwise_dot(U, V)` — `(n, d), (n, d) → (n,)`.
- `pairwise_cosine(Q, K)` — `(n, d), (m, d) → (n, m)`.

İkisinde de **döngü yok**. Kontrol, 4096 çift için `for` + `torch.dot` ile vektörize hâlini yarıştırır.

In [ ]:
%%writefile stage_e.py
import torch


def rowwise_dot(U, V):
    """(n, d), (n, d) → (n,)"""
    ...


def pairwise_cosine(Q, K):
    """(n, d), (m, d) → (n, m)"""
    ...


U, V = torch.randn(5, 3), torch.randn(5, 3)
print(rowwise_dot(U, V))
print(pairwise_cosine(U, V).shape)  # torch.Size([5, 5])

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [MIT 18.06 Linear Algebra (Strang)](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/) — CC BY-NC-SA 4.0
- [3Blue1Brown — Dot products and duality](https://www.3blue1brown.com/lessons/dot-products) — ©
- [Mathematics for Machine Learning, Bölüm 3 (Analytic Geometry)](https://mml-book.github.io/) — © (PDF ücretsiz)
- [Dive into Deep Learning §2.3.10 Dot Products](https://d2l.ai/chapter_preliminaries/linear-algebra.html) — CC BY-SA 4.0 / MIT-0